In [1]:
import json
import numpy as np
import datasets

with open("marco_qrels.json") as f:
    qrels = json.load(f)
    
ds = datasets.load_from_disk("msmarco_raw")

/home/zhichaog/anaconda3/envs/neural-sparse-mdbert/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("q_ids.json") as f:
    q_ids = json.load(f)
    
with open("d_ids.json") as f:
    d_ids = json.load(f)

In [4]:
# load scores


score_bge_gemma2 = np.load("scores/msmarco_raw/bge_gemma2_reranker_scores.npy")
score_minilm = np.load("scores/msmarco_raw/minilm_scores.npy").astype(np.float64)
score_3 = np.load("scores/msmarco_raw/llm-3_scores.npy").astype(np.float64)
score_bge_lw = np.load("scores/msmarco_raw/bge_lw_reranker_scores.npy").astype(np.float64)
# score_4 = np.load("scores/msmarco_raw/llm-4_scores.npy").astype(np.float64)
# score_5 = np.load("scores/msmarco_raw/llm-5_scores.npy").astype(np.float64)
# score_6 = np.load("scores/msmarco_raw/llm-6_scores.npy").astype(np.float64)
score_dense = np.load("scores/msmarco_raw/gte-bi_scores.npy").astype(np.float64)
score_sparse = np.load("scores/msmarco_raw/sparse-v1_scores.npy").astype(np.float64)
# score_qwen = np.load("scores/msmarco_raw/qwen3_reranker_scores.npy").astype(np.float64)
# score_qwen_06 = np.load("scores/msmarco_raw/qwen3_reranker_0.6B_scores.npy").astype(np.float64)

In [19]:
score_t5 = np.load("scores/msmarco_raw/t5_scores.npy").astype(np.float64)
print(eval_score(score_t5, sample_num=2000))

({'NDCG@10': 0.41448}, {'MAP@10': 0.337}, {'Recall@10': 0.65367}, {'P@10': 0.069})


In [5]:
from tqdm import tqdm
from beir.retrieval.evaluation import EvaluateRetrieval

def eval_score(scores, return_run_res = False, sample_num=1000000):
    run_res = dict(zip(q_ids[:sample_num], map(dict, map(zip, d_ids[:sample_num], scores[:sample_num]))))
    if return_run_res:
        return run_res
    else:
        return EvaluateRetrieval.evaluate(qrels, run_res, [10])
    
def min_max_norm(x):
    min_val = x.min(axis=-1, keepdims=True)
    max_val = x.max(axis=-1, keepdims=True)
    return (x - min_val) / (max_val - min_val + 1e-8)  # 加上小量避免除零

def combine(*scores, weights=None):
    if weights is None:
        weights = [1]*len(scores)
    scores = [min_max_norm(score)*weights[i] for i,score in enumerate(scores)]
    scores = np.sum(scores,axis=0)/sum(weights)
    return scores

In [7]:
print(eval_score(score_bge_gemma2, sample_num=2000))
print(eval_score(score_bge_lw, sample_num=2000))
print(eval_score(score_minilm, sample_num=2000))
print(eval_score(score_3, sample_num=2000))
# print(eval_score(score_4, sample_num=2000))
# print(eval_score(score_5, sample_num=2000))
# print(eval_score(score_6, sample_num=2000))
print(eval_score(score_dense, sample_num=2000))
print(eval_score(score_sparse, sample_num=2000))
# print(eval_score(score_qwen, sample_num=2000))
# print(eval_score(score_qwen_06, sample_num=2000))

({'NDCG@10': 0.48027}, {'MAP@10': 0.40239}, {'Recall@10': 0.719}, {'P@10': 0.07605})
({'NDCG@10': 0.45196}, {'MAP@10': 0.3759}, {'Recall@10': 0.68608}, {'P@10': 0.07235})
({'NDCG@10': 0.44527}, {'MAP@10': 0.36964}, {'Recall@10': 0.67696}, {'P@10': 0.0715})
({'NDCG@10': 0.45896}, {'MAP@10': 0.38254}, {'Recall@10': 0.69358}, {'P@10': 0.07315})
({'NDCG@10': 0.43465}, {'MAP@10': 0.35542}, {'Recall@10': 0.67958}, {'P@10': 0.0714})
({'NDCG@10': 0.43073}, {'MAP@10': 0.35267}, {'Recall@10': 0.67133}, {'P@10': 0.0709})


In [10]:
score = combine(score_sparse, score_dense, score_minilm, score_3, score_bge_gemma2, weights=[1,1,1,2,2])
print(eval_score(score))

({'NDCG@10': 0.48161}, {'MAP@10': 0.40605}, {'Recall@10': 0.71451}, {'P@10': 0.07452})


In [7]:
score = combine(score_sparse, score_dense, score_minilm, score_3, score_bge_gemma2, weights=[1,1,1,2,3])
print(eval_score(score))

({'NDCG@10': 0.48389}, {'MAP@10': 0.40839}, {'Recall@10': 0.71651}, {'P@10': 0.07473})


In [8]:
score = combine(score_sparse, score_dense, score_minilm, score_3, score_bge_gemma2, weights=[1,1,1,2,4])
print(eval_score(score))

({'NDCG@10': 0.4852}, {'MAP@10': 0.40972}, {'Recall@10': 0.7177}, {'P@10': 0.07486})


In [9]:
score = combine(score_sparse, score_dense, score_minilm, score_3, score_bge_gemma2, weights=[1,1,1,2,5])
print(eval_score(score))

({'NDCG@10': 0.4859}, {'MAP@10': 0.4105}, {'Recall@10': 0.71814}, {'P@10': 0.07491})


In [21]:
score = combine(score_sparse, score_dense, score_minilm, score_3, score_bge_gemma2, weights=[1,1,1,2,5])
print(eval_score(score, sample_num=2000))

score = combine(score_sparse, score_dense, score_t5,score_minilm, score_3, score_bge_gemma2, weights=[1,1,1,1,2,5])
print(eval_score(score, sample_num=2000))

({'NDCG@10': 0.49383}, {'MAP@10': 0.41768}, {'Recall@10': 0.72662}, {'P@10': 0.0768})
({'NDCG@10': 0.4918}, {'MAP@10': 0.41453}, {'Recall@10': 0.72862}, {'P@10': 0.077})


In [6]:
score = combine(score_sparse, score_dense, score_minilm, score_3, score_bge_gemma2, weights=[1,1,1,2,4])
print(eval_score(score, sample_num=2000))
score = score*30
ds = ds.remove_columns(['q_id', 'd_ids', 'dynamo_q_id', 'dynamo_d_ids'])
ds = ds.add_column('scores', score.tolist())
ds.save_to_disk("msmarco_llm_unrank")

({'NDCG@10': 0.49343}, {'MAP@10': 0.41749}, {'Recall@10': 0.72537}, {'P@10': 0.07665})


Saving the dataset (37/37 shards): 100%|██████████| 502939/502939 [00:28<00:00, 17959.95 examples/s]


In [11]:
score = combine(score_sparse, score_dense, score_minilm, score_3, score_bge_gemma2, weights=[1,1,1,2,2])
print(eval_score(score, sample_num=2000))
score = score*30
ds = ds.remove_columns(['q_id', 'd_ids', 'dynamo_q_id', 'dynamo_d_ids'])
ds = ds.add_column('scores', score.tolist())
ds.save_to_disk("msmarco_llm_unrank_2")

({'NDCG@10': 0.49038}, {'MAP@10': 0.41494}, {'Recall@10': 0.72104}, {'P@10': 0.07625})


Saving the dataset (37/37 shards): 100%|██████████| 502939/502939 [00:43<00:00, 11436.61 examples/s]


In [15]:
score = combine(score_sparse, score_dense, score_minilm, score_3, score_bge_lw, score_bge_gemma2, weights=[1,1,1,2,2,5])
print(eval_score(score, sample_num=20000))
score = score*30
ds = ds.remove_columns(['q_id', 'd_ids', 'dynamo_q_id', 'dynamo_d_ids'])
ds = ds.add_column('scores', score.tolist())
ds.save_to_disk("msmarco_llm_unrank_3")

({'NDCG@10': 0.48987}, {'MAP@10': 0.41325}, {'Recall@10': 0.72542}, {'P@10': 0.07591})


Saving the dataset (37/37 shards): 100%|██████████| 502939/502939 [01:31<00:00, 5492.43 examples/s] 


In [30]:
ds[0]["scores"][:5]

[28.59523069067718,
 27.512172222757826,
 26.765527136662673,
 26.57737099865236,
 21.289544488765397]